# 疑似データ（PerSearch の camera 再現）の準備

疑似データ（PerSearch の camera 再現ベンチマーク結果）を、camera 実データと直接比較できる
「生の0/1・ドル価格」符号化の long CSV に変換する。

`data/persearch_camera.xlsx`（PerSearch で `camera` と同じ7属性・価格5水準
（$79/$129/$179/$229/$279）の調査を実施した結果）の「回答データ（Long形式）」
シートを読み、`bayesm::camera` の生の X 符号化（canon/sony/nikon/panasonic の
4ダミー・pixels/zoom/video/swivel/wifi の2値ダミー・price を100ドル単位の
連続値）に揃えた long CSV を出力する。

読み込み・変換・検証のロジックは `conjoint_lib.prepare` にまとまっている。日本語ラベルの
対応表は `conjoint_lib.schema` にあり、PerSearch の出力文言が対応表とズレていると
`ValueError` になる（過去に「動画撮影」列の文言のズレに気づかず該当列が全行0になった
事故があったため、意図的に検出するようにしてある）。

参照: `prepare_camera.ipynb`（camera 側の同形式変換）、`hb_diag.ipynb`（本notebookの
出力を読んで対角共分散の階層ベイズMNLを推定する）。


In [1]:
from __future__ import annotations

import pandas as pd

from conjoint_lib import schema
from conjoint_lib.paths import DATA_DIR, RESULTS_DIR
from conjoint_lib.prepare import load_persearch_sheet, run_prepare_persearch


## 属性・水準の対応

PerSearch側の日本語ラベル -> raw 0/1・camera_long.csv と同じカテゴリ文字列の対応表（`conjoint_lib.schema.CAMERA_STUDY` から導出。2値属性のみ表示）。

In [2]:
maps = pd.DataFrame(
    [
        (attr.ja_column, lvl.ja_labels[0], 0 if lvl.raw_column is None else 1, lvl.label)
        for attr in schema.CAMERA_STUDY.categorical_attributes()
        if len(attr.levels) == 2
        for lvl in attr.levels
    ],
    columns=["column(ja)", "level(ja)", "raw(0/1)", "label(camera_long.csv 表記)"],
)
maps


,column(ja),level(ja),raw(0/1),label(camera_long.csv 表記)
0,画素数,標準画素数,0,低画素
1,画素数,高画素数,1,高画素
2,ズーム,標準ズーム,0,標準ズーム
3,ズーム,高倍率ズーム,1,高倍率ズーム
4,動画撮影,動画非対応,0,動画機能なし
5,動画撮影,動画対応,1,動画機能あり
6,可動式モニタ,可動式モニタなし,0,回転式液晶なし
7,可動式モニタ,可動式モニタあり,1,回転式液晶あり
8,Wi-Fi,Wi-Fi非対応,0,Wi-Fiなし
9,Wi-Fi,Wi-Fi対応,1,Wi-Fiあり


## ラベル文言の確認

xlsx を PerSearch で再生成・差し替えたときは、実行前に各列の実際の文言が上の対応表と
一致しているか確認する（対応表とズレたまま実行すると、該当列が `ValueError` で止まる）。


In [3]:
raw_sheet = load_persearch_sheet(DATA_DIR / "persearch_camera.xlsx")

for attr in schema.CAMERA_STUDY.categorical_attributes():
    print(attr.ja_column, sorted(raw_sheet[attr.ja_column].dropna().unique().tolist()))


ブランド ['キヤノン', 'ソニー', 'ニコン', 'パナソニック']
画素数 ['標準画素数', '高画素数']
ズーム ['標準ズーム', '高倍率ズーム']
動画撮影 ['動画対応', '動画非対応']
可動式モニタ ['可動式モニタあり', '可動式モニタなし']
Wi-Fi ['Wi-Fi対応', 'Wi-Fi非対応']


## 実行

In [4]:
run_prepare_persearch(xlsx_path=DATA_DIR / "persearch_camera.xlsx", out_dir=RESULTS_DIR)


wrote 53120 rows to C:\Users\kiskl\OneDrive\デスクトップ\projects\qiita_articles\persearch_demo\results\persearch_camera_raw_long.csv
respondents: 332, tasks/respondent: 32, alts/task: 5
none choice rate: 0.22317394578313254
price levels: [np.float64(0.0), np.float64(0.79), np.float64(1.29), np.float64(1.79), np.float64(2.29), np.float64(2.79)]


wrote 53120 rows to C:\Users\kiskl\OneDrive\デスクトップ\projects\qiita_articles\persearch_demo\results\persearch_camera_long.csv


{'raw_long': WindowsPath('C:/Users/kiskl/OneDrive/デスクトップ/projects/qiita_articles/persearch_demo/results/persearch_camera_raw_long.csv'),
 'long': WindowsPath('C:/Users/kiskl/OneDrive/デスクトップ/projects/qiita_articles/persearch_demo/results/persearch_camera_long.csv')}